# **AutoGen:-**
- AutoGen is an open-source framework by Microsoft Research for building multi-agent AI systems.
Instead of using one AI for everything, AutoGen allows multiple AI agents to work together, communicate, share tasks, and solve complex problems automatically.

## Difference: AutoGen vs. LangChain vs. LangGraph
**AutoGen →** Multiple AI agents communicate and collaborate like a team to solve tasks. It provides medium to high control over conversations and agent interactions. Best for research assistants, coding agents, task automation, and multi-agent applications.

**LangChain →** Uses prompts, tools, retrievers, and chains to build LLM applications quickly. It provides medium control and offers many ready-made components. Best for chatbots, RAG applications, document Q&A, and rapid AI development.

**LangGraph →** Uses a graph-based workflow with nodes, edges, loops, and branching paths. It provides very high control over state management and execution flow. Best for production AI systems that need checkpoints, retries, approvals, and complex workflows.

**In short, AutoGen is for collaboration, LangChain is for rapid development, and LangGraph is for workflow control.**


**Common Uses of AutoGen:-**
"AutoGen is commonly used for automated coding and debugging, research and report generation, business workflow automation, data analysis, and customer support systems. Its main advantage is that multiple AI agents can collaborate like a team, where each agent has a specific role and works together to solve complex tasks efficiently."



### **Agents:-**
**ConversableAgent ->** Base/parent agent. Sabhi agents isi se bante hain aur messages send/receive karte hain.

**AssistantAgent ->** AI brain. Planning karta hai, code likhta hai, answers generate karta hai, aur debugging me help karta hai.

**UserProxyAgent ->** User ka representative. Human input leta hai aur code execute karke results wapas bhejta hai.

**GroupChat & GroupChatManager ->** Multiple agents ko manage karta hai aur decide karta hai ki kaun agent kab bolega.

**RetrieveAssistantAgent ->** Documents, PDFs, ya knowledge base se information dhoondh kar answers deta hai.

**RetrieveUserProxyAgent ->** Vector database se relevant data nikaal kar RetrieveAssistantAgent ko provide karta hai.

**TeachableAgent ->** User preferences aur past interactions yaad rakh sakta hai (long-term memory).

**MultimodalConversableAgent ->** Text ke saath images ko bhi samajh sakta hai.

**SocietyOfMindAgent ->** Multiple agents ko ek single agent ki tarah organize aur manage karta hai.

### **AutoGen Works:-**
"AutoGen works by creating multiple agents with different roles, such as a Researcher, Coder, and Reviewer. When a task is given, the agents communicate and collaborate with each other, use tools if needed, and work together to solve the problem. A human can also be involved for approval or feedback. Once the task is completed, AutoGen provides the final result."

### Easy Flow:
 **Create Agents(Researcher, Coder, or Reviewer, and connect them with LLMs and tools) -> Give Task(gole and prompt) -> Agents Collaborate -> Human Approval (Optional) -> Final Output**

In [2]:
!pip install google-generativeai
!pip install pyautogen
!pip install autogen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 13.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [autogen]


In [ ]:
import google.generativeai as genai

GEMINI_API_KEY = "AQ.................."
genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel("gemini-3.5-flash")

def gemini_chat(prompt):
    response = model.generate_content(prompt)
    return response.text


### Define Custom Gemini Agent

In [10]:
from autogen import ConversableAgent

class GeminiAgent(ConversableAgent):
    def __init__(self, name, **kwargs):
        super().__init__(name, **kwargs)

    def generate_reply(self, messages, sender, config=None):
        prompt = "\n".join([m["content"] for m in messages])
        reply = gemini_chat(prompt)
        return {"role": "assistant", "content": reply}


### Initialize All Agents

In [11]:
from autogen import UserProxyAgent

user_proxy = UserProxyAgent(name="User", human_input_mode="TERMINATE")

analyst = GeminiAgent(name="AnalystBot")
swot = GeminiAgent(name="SWOTBot")
strategist = GeminiAgent(name="StrategistBot")
critic = GeminiAgent(name="CriticBot")


In [12]:
# Business Description (Input)
business_description = """
Our company is a 2-year-old startup that provides AI-powered resume screening software for
HR teams.
We’ve gained traction in India but are struggling to expand internationally.
The product integrates with common HR tools and uses machine learning to rank candidates.
We face growing competition and challenges in compliance and data privacy in
international markets.
"""


### Chain Agents via Shared Context

In [13]:
# Step 1: Ask AnalystBot to analyze the business
messages = [{"role": "user", "content": f"Analyze the following business and identify key challenges and strengths:\n\n{business_description}"}]
analyst_output = analyst.generate_reply(messages, sender="User")["content"]
print("AnalystBot Output:\n", analyst_output)

# Step 2: Pass analyst_output to SWOTBot
messages = [{"role": "user", "content": f"Based on this analysis, provide a detailed SWOT analysis:\n\n{analyst_output}"}]
swot_output = swot.generate_reply(messages, sender="User")["content"]
print("\n SWOTBot Output:\n", swot_output)

# Step 3: Pass SWOT to StrategistBot
messages = [{"role": "user", "content": f"Based on this SWOT analysis, suggest 3 business growth strategies:\n\n{swot_output}"}]
strategies_output = strategist.generate_reply(messages, sender="User")["content"]
print("\n StrategistBot Output:\n", strategies_output)

# Step 4: Ask CriticBot to critique the strategies
messages = [{"role": "user", "content": f"Critique the following strategies and suggest improvements:\n\n{strategies_output}"}]
critic_output = critic.generate_reply(messages, sender="User")["content"]
print("\n CriticBot Output:\n", critic_output)


AnalystBot Output:
 Here is a comprehensive business analysis of your startup, broken down by key strengths, major challenges, and actionable strategic recommendations.

---

### **Key Strengths**

1. **Proven Market Traction (Proof of Concept):** 
   * Gaining traction in India—a large, complex, and highly competitive talent market—proves that the product solves a real pain point (high-volume resume screening) and delivers tangible value to HR teams.

2. **Ecosystem-Friendly Architecture (Low Friction):** 
   * Native integration with existing HR tools (ATS, HRIS like Greenhouse, Workday, or BambooHR) significantly lowers adoption friction. HR teams don't want another standalone tool; they want enhancements to their current workflow.

3. **Time-to-Value & Efficiency Focus:** 
   * Machine learning ranking directly addresses HR's biggest bottleneck: manual resume sorting. By automating candidate prioritization, you provide a clear, measurable ROI (time saved, reduced time-to-hire).

4.